# Логистическая регрессия для классификации текста

## Задача классификации текста

Классификация лежит в основе многих задач NLP: определение тональности отзыва, фильтрация спама, определение языка. 
В этом занятии мы разберём несколько фундаментальных механизмов, на которых построена логистическая регрессия и, 
как следствие, современные нейронные сети.

#### Примеры задач:
- **Это спам?**
- **Кто написал Федералистские статьи?** (Гамильтон, Мэдисон, Джей)
- **Позитивный или негативный отзыв о фильме?**
- **Моделирование языка** = классификация следующего слова

**Почему не просто правила?**

Можно писать правила вручную, например:
- если есть слово *love* → позитив

Проблемы:
- не масштабируется
- много исключений
- сложно поддерживать

**Вывод:** используем машинное обучение

## Математическая модель

**Вход:**
- Набор признаков $x$
- Набор целевых значений $y$

**Выход:**
- Функция $\phi$, где мы минимизируем некоторую *ошибку*.

### Пример

Задача: Предсказание дохода сотрудника

Цель: На основе возраста и опыта работы предсказать зарплату сотрудника.

In [ ]:
%conda install kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hussainnasirkhan/multiple-linear-regression-dataset")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
import csv

data = []

with open("data.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        data.append({
            "age": float(row["age"]),
            "experience": float(row["experience"]),
            "income": float(row["income"])
        })

def predict(row, w1, w2, b):
    return w1 * row["age"] + w2 * row["experience"] + b


def mse(w1, w2, b):
    total = 0.0
    n = len(data)

    for row in data:
        y_pred = predict(row, w1, w2, b)
        error = y_pred - row["income"]
        total += error * error

    return total / n

def grad_w1(w1, w2, b, h=1e-5):
    return (mse(w1 + h, w2, b) - mse(w1, w2, b)) / h

def grad_w2(w1, w2, b, h=1e-5):
    return (mse(w1, w2 + h, b) - mse(w1, w2, b)) / h

def grad_b(w1, w2, b, h=1e-5):
    return (mse(w1, w2, b + h) - mse(w1, w2, b)) / h

w1 = 0.0
w2 = 0.0
b = 0.0

learning_rate = 0.000001
epochs = 1000

for epoch in range(epochs):

    g1 = grad_w1(w1, w2, b)
    g2 = grad_w2(w1, w2, b)
    gb = grad_b(w1, w2, b)

    w1 -= learning_rate * g1
    w2 -= learning_rate * g2
    b  -= learning_rate * gb

    if epoch % 50 == 0:
        print("epoch:", epoch, "mse:", mse(w1, w2, b))


def predict_income(age, experience):
    return w1 * age + w2 * experience + b

print("Prediction:", predict_income(35, 10))

## Переход от регрессии к классификации

В исходной задаче мы решали задачу **регрессии** — предсказывали непрерывную величину (зарплату) на основе признаков: возраста и опыта работы.

Линейная регрессионная модель имеет вид:

$$
\hat{y} = w_1 \cdot age + w_2 \cdot experience + b
$$

где:
- $\hat{y}$ — предсказанная зарплата  
- $w_1, w_2$ — веса признаков  
- $b$ — смещение  

Для обучения модели использовалась функция ошибки **MSE (Mean Squared Error)**:

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

## Переход к задаче классификации

Теперь изменим постановку задачи. Вместо предсказания точного значения зарплаты будем определять **категорию сотрудника**, например:

- 0 — низкая зарплата  
- 1 — высокая зарплата  

Таким образом, задача становится задачей **бинарной классификации**.


### Математическая модель

Для решения задачи классификации используется **логистическая регрессия**.

Сначала, как и в линейной регрессии, вычисляется линейная комбинация признаков:

$$
z = w_1 \cdot age + w_2 \cdot experience + b
$$

### А как дальше?
Проблема: $z$ — не вероятность.
Решение: сигмоидная функция $\sigma(z) \in [0,1]$:
$$\sigma(z) = \frac{1}{1 + e^{-z}} = \frac{1}{1 + \exp(-z)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    """Сигмоидная функция"""
    return 1 / (1 + np.exp(-z))

# Визуализация
z = np.linspace(-10, 10, 100)
plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid(z), 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', label='Граница решения')
plt.xlabel('z = w·x + b'); plt.ylabel('σ(z)')
plt.title('Сигмоидная функция'); plt.grid(True, alpha=0.3)
plt.legend(); plt.show()

Итого применим сигмойду и будем от 0 до 1. Сигмоидная функция:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Итоговое предсказание:

$$
\hat{y} = \sigma(z)
$$

### И как с этим работать?
$$P(y=1|x) = \sigma(w \cdot x + b)$$
$$P(y=0|x) = 1 - \sigma(w \cdot x + b)$$
> Свойство: $1 - \sigma(x) = \sigma(-x)$

### Обучение: Кросс-энтропийная потеря
Цель: минимизировать расстояние между $\hat{y}$ и $y$

Вывод для одного наблюдения:
1. $p(y|x) = \hat{y}^y (1-\hat{y})^{1-y}$ (Бернулли)
2. $\log p(y|x) = y \log \hat{y} + (1-y) \log(1-\hat{y})$
3. $L_{CE} = -[y \log \hat{y} + (1-y) \log(1-\hat{y})]$

### Почему не используется MSE

MSE не подходит для классификации, потому что:
- плохо работает с вероятностями  
- приводит к медленному обучению  
- не учитывает вероятностную природу задачи  

Log Loss:
- лучше подходит для вероятностей  
- обеспечивает более стабильную сходимость  

### Сравнение задач

| Характеристика | Регрессия | Классификация |
|----------------|----------|--------------|
| Выход модели   | Число     | Вероятность (0–1) |
| Функция потерь | MSE       | Log Loss |
| Пример задачи  | Зарплата  | Класс (высокая/низкая) |

### Интерпретация результата

Значение $\hat{y}$ интерпретируется как вероятность принадлежности к классу 1:

- если $\hat{y} \geq 0.5$ → класс 1  
- если $\hat{y} < 0.5$ → класс 0  


### Функция потерь

В задаче классификации используется логарифмическая функция потерь (Binary Cross-Entropy):

$$
L = -\frac{1}{n} \sum_{i=1}^{n} \left[
y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i)
\right]
$$



### Вывод

Переход от регрессии к классификации включает:

1. Использование сигмоидной функции для получения вероятности  
2. Замену функции потерь на Log Loss  
3. Интерпретацию результата как класса, а не числа  

## Типы классификаторов:
- Логистическая регрессия (рассматриваем сегодня)
- Наивный Байес
- Нейронные сети
- k-ближайших соседей
- Большие языковые модели (LLMs)

## Четыре компонента логистической регрессии:
1. **Представление признаков**: $x = [x_1, x_2, ..., x_n]$
2. **Функция классификации**: $p(y|x)$ через сигмоиду/софтмакс
3. **Целевая функция**: кросс-энтропийная потеря
4. **Оптимизация**: стохастический градиентный спуск (SGD)

## Две фазы логистической регрессии
- **Обучение**: учим веса $w$ и смещение $b$ через SGD и cross-entropy loss
- **Тестирование**: для $x$ вычисляем $p(y|x)$ и возвращаем метку с большей вероятностью

Поиграемся ещё раз со старой задачей: 

In [ ]:
print(data)

In [ ]:
import random
import math

data_raw = data

incomes = [row["income"] for row in data_raw]
median_income = sorted(incomes)[len(incomes) // 2]

data = []

for row in data_raw:
    noise = random.randint(-10000, 10000)  # стохастичность
    noisy_income = row["income"] + noise

    label = 1 if noisy_income > median_income else 0

    data.append({
        "age": row["age"],
        "experience": row["experience"],
        "label": label
    })

w1 = 0.0
w2 = 0.0
b = 0.0

learning_rate = 0.001
epochs = 5000

n = len(data)

def sigmoid(z):
    return 1 / (1 + math.exp(-z))


for epoch in range(epochs):

    dw1 = 0.0
    dw2 = 0.0
    db = 0.0

    loss = 0.0

    for row in data:
        x1 = row["age"]
        x2 = row["experience"]
        y = row["label"]

        # линейная часть
        z = w1 * x1 + w2 * x2 + b
        y_pred = sigmoid(z)

        # log loss (устойчивый вариант)
        y_pred = max(1e-15, min(1 - 1e-15, y_pred))
        loss += -(y * math.log(y_pred) + (1 - y) * math.log(1 - y_pred))

        # градиенты
        error = y_pred - y
        dw1 += error * x1
        dw2 += error * x2
        db += error

    # обновление весов
    w1 -= learning_rate * (1 / n) * dw1
    w2 -= learning_rate * (1 / n) * dw2
    b  -= learning_rate * (1 / n) * db

    if epoch % 500 == 0:
        print("epoch:", epoch, "loss:", loss / n)


def predict(age, experience):
    z = w1 * age + w2 * experience + b
    return 1 if sigmoid(z) >= 0.5 else 0

# пример
print("Class prediction:", predict(35, 10))

А теперь многомерная:

In [ ]:
import csv
import random
import math

raw_data = []

with open("data.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw_data.append({
            "age": float(row["age"]),
            "experience": float(row["experience"]),
            "income": float(row["income"])
        })


def mean_std(values):
    m = sum(values) / len(values)
    var = sum((x - m) ** 2 for x in values) / len(values)
    return m, math.sqrt(var)

ages = [r["age"] for r in raw_data]
exps = [r["experience"] for r in raw_data]

age_mean, age_std = mean_std(ages)
exp_mean, exp_std = mean_std(exps)

def normalize(x, mean, std):
    return (x - mean) / (std + 1e-8)

incomes = sorted([r["income"] for r in raw_data])

q1 = incomes[len(incomes)//4]
q2 = incomes[len(incomes)//2]
q3 = incomes[(3*len(incomes))//4]

data = []

for r in raw_data:
    x1 = normalize(r["age"], age_mean, age_std)
    x2 = normalize(r["experience"], exp_mean, exp_std)

    income = r["income"]

    if income <= q1:
        label = 0
    elif income <= q2:
        label = 1
    elif income <= q3:
        label = 2
    else:
        label = 3

    data.append((x1, x2, label))


K = 4  # классы

W = [[random.uniform(-0.01, 0.01) for _ in range(2)] for _ in range(K)]
b = [0.0 for _ in range(K)]


def softmax(z):
    max_z = max(z)
    exp_z = [math.exp(i - max_z) for i in z]
    s = sum(exp_z)
    return [i / s for i in exp_z]

def forward(x1, x2):
    z = []
    for k in range(K):
        z_k = W[k][0] * x1 + W[k][1] * x2 + b[k]
        z.append(z_k)
    return softmax(z)

def loss():
    total = 0.0

    for x1, x2, y in data:
        p = forward(x1, x2)
        total += -math.log(p[y] + 1e-15)

    return total / len(data)

lr = 0.01
epochs = 50

for epoch in range(epochs):

    random.shuffle(data)

    for x1, x2, y in data:

        # forward
        p = forward(x1, x2)

        # gradient for softmax + cross-entropy:
        # dp_k = p_k - y_k

        for k in range(K):
            indicator = 1 if k == y else 0
            error = p[k] - indicator

            # update weights
            W[k][0] -= lr * error * x1
            W[k][1] -= lr * error * x2
            b[k]    -= lr * error

    if epoch % 5 == 0:
        print("epoch:", epoch, "loss:", loss())

def predict(age, experience):
    x1 = normalize(age, age_mean, age_std)
    x2 = normalize(experience, exp_mean, exp_std)

    p = forward(x1, x2)
    return p.index(max(p))

print("Prediction:", predict(35, 10))

## Масштабирование входных признаков (Feature Scaling)

### Зачем нужно масштабирование признаков

Когда разные признаки входных данных имеют **очень разные диапазоны значений**, это может ухудшать обучение моделей машинного обучения.

Примеры проблем:

- один признак: возраст (0–100)
- другой признак: доход (0–1 000 000)

Модель начинает "перекосно" учитывать признаки с большими значениями.


### Стандартизация (Z-score normalization)

Один из самых распространённых способов масштабирования — это стандартизация.

Она приводит данные к:

- среднему значению = 0
- стандартному отклонению = 1

#### Среднее значение признака:

$$ \mu_i = \frac{1}{m} \sum_{j=1}^{m} x_i^{(j)} $$

#### Стандартное отклонение:

$$ \sigma_i = \sqrt{\frac{1}{m} \sum_{j=1}^{m} (x_i^{(j)} - \mu_i)^2} $$


#### Преобразование (Z-score):

$$ x'_i = \frac{x_i - \mu_i}{\sigma_i} $$


#### Нормализация в диапазон [0, 1]

Альтернативный способ масштабирования — min-max нормализация:

$$ x'_i = \frac{x_i - \min(x_i)}{\max(x_i) - \min(x_i)} $$


### Зачем это нужно

Масштабирование признаков полезно, потому что:

- ускоряет градиентный спуск
- делает обучение более стабильным
- предотвращает доминирование одних признаков над другими

### Пример

Если один признак имеет значения:

- возраст: [20, 25, 30]

а другой:

- доход: [10 000, 50 000, 100 000]

без масштабирования модель будет в основном ориентироваться на доход.

### Масштабирование и нейронные сети

В больших нейронных сетях масштабирование особенно важно, потому что:

- ускоряет сходимость градиентного спуска
- уменьшает вероятность взрыва или исчезновения градиентов

### Логарифмическое масштабирование (для NLP и счётчиков)

Для текстовых данных часто используют логарифмирование.

Это связано с тем, что данные (например, частоты слов) часто имеют **Zipf-распределение**.

Пример:

- частота слов
- биграммы
- счётчики встречаемости

#### Логарифмическое преобразование:

$$ x' = \log(x + 1) $$

### Итог

Основные методы масштабирования:

- Z-score стандартизация
- Min-Max нормализация
- логарифмическое преобразование

### Вывод

Масштабирование признаков — это важный этап предобработки данных, который:

- улучшает обучение моделей
- ускоряет оптимизацию
- делает признаки сопоставимыми по масштабу

## Формулировка задачи классификации для текста

Теперь переходим к задаче классификации текста. Наша целевая переменная становится дискретной:

$$ y \in \{0, 1, ..., K\}$$

Пока для простоты:

$$ y \in \{0, 1\}$$

## Основная проблема: текст нельзя подать в модель напрямую

Модели машинного обучения работают с числами, поэтому текст необходимо преобразовать в числовое представление.

**Для этого есть куча методов:**
- Bag of Words (мешок слов)
- TF-IDF
- Word embeddings (Word2Vec, GloVe)
- Transformer embeddings

После преобразования текст становится вектором:

$$ text \rightarrow \mathbf{x} \in \mathbb{R}^n $$

## Новая модель: логистическая регрессия

Для бинарной классификации используется сигмоидная функция:

$$ p = \sigma(\mathbf{w} \cdot \mathbf{x} + b) $$

где:

- $p$ — вероятность класса 1
- $\sigma(z) = \frac{1}{1 + e^{-z}}$

### Функция потерь

В отличие от регрессии (MSE), для классификации обычно используют (Cross-Entropy Loss):

$$ L = - \left( y \log(p) + (1 - y)\log(1 - p) \right) $$

Она лучше отражает вероятностную природу задачи.

### Итоговый переход

| Регрессия | Классификация текста |
|----------|----------------------|
| income (число) | класс (0/1/…K) |
| MSE | Cross-Entropy |
| линейный выход | sigmoid / softmax |
| числовые признаки | текст → вектор |

### Вывод

Переход к классификации текстов состоит из двух ключевых шагов:

1. **Преобразование текста в числовой вектор**
2. **Замена регрессионной модели на классификационную**

Таким образом, общая схема становится:

$$ text \rightarrow vector \rightarrow model \rightarrow class $$

Это базовая идея всех современных NLP-моделей, включая нейросети и трансформеры.

## Пример: классификация тональности
Текст: *"It's hokey... the cast is great... It sucked me in"*

| Признак | Значение |
|---------|----------|
| $x_1$ = count(positive) | 3 |
| $x_2$ = count(negative) | 2 |
| $x_3$ = есть "no" | 1 |
| $x_4$ = местоимения | 3 |
| $x_5$ = есть "!" | 0 |
| $x_6$ = log(word_count) | 4.19 |

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Данные из примера
w = np.array([2.5, 5.0, 1.2, 0.5, 2.0, 0.7])
x = np.array([3, 2, 1, 3, 0, 4.19])
b = 0.1

z = np.dot(w, x) + b
p_pos = sigmoid(z)
p_neg = 1 - p_pos

print(f"z = {z:.3f}")
print(f"P(позитивный | x) = {p_pos:.2f}")
print(f"P(негативный | x) = {p_neg:.2f}")
print(f"Предсказание: {'позитивный' if p_pos > 0.5 else 'негативный'}")

In [ ]:
def cross_entropy_loss(y_true, y_pred, eps=1e-15):
    """Кросс-энтропийная потеря"""
    y_pred = np.clip(y_pred, eps, 1-eps)
    return -(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))

# Пример: p_pos = 0.70 из предыдущего расчёта
loss_correct = cross_entropy_loss(1, p_pos)  # y=1, модель права
loss_wrong = cross_entropy_loss(0, p_pos)    # y=0, модель ошибается

print(f"Потеря (модель права): {loss_correct:.2f}")
print(f"Потеря (модель ошибается): {loss_wrong:.2f}")
print("✓ Потеря меньше, когда модель права!")